# Getting started: Brazilian benefit-card transactions

A synthetic year of benefit-card purchases (meal, food, mobility, multi-benefit, incentive) answered by a simulated issuer. This notebook loads the tables, then looks at what the issuer approved and declined, when people spend, the balance ledger and the labelled fraud.

Every record is synthetic. The data dictionary is `data_dictionary.md`; the code that builds it: [github.com/Nic-Soares/corporate-benefits-transactions](https://github.com/Nic-Soares/corporate-benefits-transactions).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# On Kaggle the files are under /kaggle/input/<dataset>/; locally, in data/dataset/
candidates = list(Path("/kaggle/input").rglob("fact_transactions.parquet")) or list(
    Path("..").rglob("data/dataset/fact_transactions.parquet")
)
DATA = candidates[0].parent
print(DATA)
sorted(p.name for p in DATA.iterdir())

## The tables

A star schema: `fact_transactions` (one row per purchase attempt, with the issuer's answer) and `fact_balance_movements` (every credit, debit and reversal of a wallet), dimensions (`dim_*`) and the rules behind each decline (`ref_*`). The ids join them.

In [ ]:
tx = pd.read_parquet(
    DATA / "fact_transactions.parquet",
    columns=[
        "id",
        "card_id",
        "company_id",
        "modality_id",
        "wallet_id",
        "transaction_type",
        "establishment_id",
        "amount",
        "date",
        "capture_type",
        "response_code",
        "decline_reason",
        "is_fraud",
        "fraud_scenario",
        "disputed",
    ],
)
tx["amount"] = tx["amount"].astype(float)
tx["approved"] = tx["response_code"].eq("00")
print(f"{len(tx):,} attempts from {tx['date'].min():%Y-%m-%d} to {tx['date'].max():%Y-%m-%d}")
tx.head()

In [ ]:
shops = pd.read_parquet(DATA / "dim_establishments.parquet", columns=["id", "category", "state"])
codes = pd.read_parquet(DATA / "ref_response_codes.parquet")
tx = tx.merge(shops.rename(columns={"id": "establishment_id"}), on="establishment_id", how="left")
codes

## Approved and declined

About 85–90% of attempts are approved each month. Declines carry an ISO 8583 code and a more specific `decline_reason`; most come from a wallet without balance.

In [ ]:
attempts = tx[tx["transaction_type"] != "reversal"]
monthly = attempts.groupby(attempts["date"].dt.to_period("M")).agg(
    attempts=("id", "size"), approved=("approved", "mean")
)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly.index.astype(str), monthly["attempts"] / 1000, color="#d9eb99", edgecolor="#566422")
ax.set_ylabel("attempts (thousands)")
ax2 = ax.twinx()
ax2.plot(monthly.index.astype(str), monthly["approved"] * 100, color="#3a665d", marker="o")
ax2.set_ylim(80, 92)
ax2.set_ylabel("approved (%)")
ax.set_title("Attempts by month and the share approved")
plt.setp(ax.get_xticklabels(), rotation=45)
plt.show()
monthly.assign(approved=lambda d: (d["approved"] * 100).round(1))

In [ ]:
declines = attempts.loc[~attempts["approved"], "decline_reason"].value_counts(normalize=True)
(declines.head(10) * 100).round(1).rename("% of declines")

## Which wallet pays

On the issuer's proprietary network (`OWN`) the shop's CNAE decides which wallet it takes; on a brand (Visa, Mastercard, Elo) the MCC does. Restaurants are paid by the meal wallet, supermarkets by food, fuel by mobility; the gift wallet takes anything on a brand card.

In [ ]:
paid = attempts[attempts["approved"]]
pd.crosstab(paid["category"], paid["wallet_id"], normalize="index").mul(100).round(1)

## When people spend

Restaurants peak at lunch and sell less on weekends; supermarkets peak on Saturday; spending jumps on the credit day (the 1st), on holidays and on Christmas Eve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for category, color in [("restaurant", "#566422"), ("supermarket", "#3a665d"), ("fuel", "#77786a")]:
    part = tx[tx["category"] == category]
    part.groupby(part["date"].dt.hour).size().pipe(lambda s: s / s.sum() * 100).plot(
        ax=axes[0], color=color, label=category
    )
    part.groupby(part["date"].dt.day_of_week).size().pipe(lambda s: s / s.sum() * 100).plot(
        ax=axes[1], color=color, marker="o", label=category
    )
axes[0].set(title="Share of the day's attempts by hour", xlabel="hour", ylabel="%")
axes[1].set(title="Share of the week by weekday", xlabel="Monday = 0", ylabel="%")
axes[0].legend()
plt.show()

In [ ]:
daily = tx.groupby(tx["date"].dt.date).size()
ax = daily.plot(figsize=(13, 4), color="#566422", title="Attempts per day")
for day, label in [("2025-12-24", "Christmas Eve"), ("2026-04-03", "Good Friday")]:
    ax.annotate(
        label,
        (pd.Timestamp(day), daily[pd.Timestamp(day).date()]),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
    )
plt.show()

## The balance ledger

Each wallet's movements chain: a movement starts where the previous one ended (`sequence_number` orders them). The monthly credit lands on the 1st; purchases spend it.

In [ ]:
ledger = pd.read_parquet(DATA / "fact_balance_movements.parquet")
wallet = ledger["card_wallet_id"].value_counts().index[100]
one = ledger[ledger["card_wallet_id"] == wallet].sort_values("sequence_number")
columns = [
    "operation_date",
    "operation_type",
    "operation_amount",
    "previous_balance",
    "posterior_balance",
]
one[columns].head(12)

In [ ]:
# Every movement starts where the previous one of the same wallet ended
ledger = ledger.sort_values(["card_wallet_id", "sequence_number"])
before = ledger.groupby("card_wallet_id")["posterior_balance"].shift()
chained = (before.isna() | (before == ledger["previous_balance"])).mean()
print(f"{chained:.2%} of the movements chain")

## Fraud

About 0.5% of the attempts are fraud, labelled with how it happened. Most approved fraud is disputed by the cardholder; a few legit purchases are disputed too (friendly fraud) — the noise a fraud model has to live with.

In [ ]:
fraud = tx[tx["is_fraud"]]
summary = fraud.groupby("fraud_scenario").agg(
    attempts=("id", "size"), approved=("approved", "mean"), avg_amount=("amount", "mean")
)
summary.assign(
    approved=lambda d: (d["approved"] * 100).round(1),
    avg_amount=lambda d: d["avg_amount"].round(2),
)

In [ ]:
pd.crosstab(tx["is_fraud"], tx["disputed"], margins=True)

## Next

- Model the declines: which features predict `insufficient_balance` before the issuer answers?
- Fraud detection with label noise: `is_fraud` is the truth, `disputed` is what an issuer would see.
- Join `dim_cardholders`, `dim_companies` and `dim_terminals` for geography and segments.

Every column is described in `data_dictionary.md`; `quality_report.md` lists the checks the data passed; `dataset_metadata.json` says how it was made.